# Chapter 07. 실행되는 모델에서 검증된 모델로 — 한국어 텍스트 분류·추천 개선하기

Chapter 06에서는 TF-IDF, Naive Bayes, 코사인 유사도 분석을 Streamlit 앱으로 연결했습니다.

하지만 **앱이 실행된다는 것과 분석 결과가 좋은 것은 같은 문제가 아닙니다.**

이번 Chapter에서는 다음 두 문제를 실제 숫자로 확인하고 개선합니다.

- 일부 제목의 **분야 예측이 직관과 다르게 나오는 문제**
- 코사인 유사도가 **0인 책도 무조건 Top 5에 들어가는 문제**

이번 Chapter의 전체 흐름은 다음과 같습니다.

**개선 데이터 확인 → Baseline 측정 → Kiwi 형태소 분석 → 작은 단위의 필터링 실험 → 같은 Train/Test 조건에서 Before/After 비교 → 추천 로직 개선 → Streamlit 앱 반영 → 최종 검증**

이번 실습의 핵심은 “더 복잡한 모델”이 아닙니다.

> **기준점을 만들고, 한 번에 작은 변경을 적용하고, 같은 조건에서 다시 측정하는 과정**을 익히는 것이 목표입니다.

## 학습 목표

이번 Chapter가 끝나면 다음을 설명할 수 있어야 합니다.

- 개선 전에 Baseline이 왜 필요한지
- 동일한 Train/Test split을 유지해야 하는 이유
- Kiwi 형태소 분석 결과를 직접 확인하는 방법
- NNG / NNP / SL 품사를 선택한 이유
- 짧은 토큰·숫자 토큰·불용어를 단계별로 필터링하는 방법
- Accuracy와 Macro F1을 Before/After로 비교하는 방법
- 특정 예제 하나가 아니라 전체 test 결과로 개선을 판단해야 하는 이유
- 유사도 0인 추천을 제거해야 하는 이유
- 추천 결과가 5권 미만 또는 0권이어도 정상일 수 있는 이유
- 개선된 전처리와 추천 로직을 Streamlit 앱에 연결하는 방법

## 실습 1. 실행 환경과 필요한 파일 확인하기

### AI에게 질문

> Chapter 07에서 books_improved.csv와 kiwipiepy를 사용하려고 합니다.  
> 현재 Notebook Python 경로와 버전, pandas/scikit-learn/kiwipiepy 설치 여부, CSV 존재 여부를 한 번에 확인하는 코드를 작성해 주세요.

### 초보자 상세 설명

앞 Chapter에서 터미널의 Python 3.12와 Notebook의 Python 3.14가 달라 패키지가 설치되어 있는데도 import가 안 되는 문제가 있었습니다.

그래서 이번에도 **코드보다 먼저 현재 Notebook이 어느 Python을 사용하는지 확인**합니다.

또한 이번 Chapter의 기준 데이터는 `books_improved.csv`입니다. Notebook이 실행되는 현재 작업 폴더에 따라 상대경로가 달라질 수 있으므로 두 위치를 모두 확인하도록 작성합니다.

In [ ]:
# 현재 Notebook Python을 확인합니다.
import sys
from pathlib import Path

print("Python 실행 파일:")
print(sys.executable)

print("\nPython 버전:")
print(sys.version)

# 기본 패키지를 확인합니다.
import numpy as np
import pandas as pd
import sklearn

print("\nnumpy 버전:", np.__version__)
print("pandas 버전:", pd.__version__)
print("scikit-learn 버전:", sklearn.__version__)

# Kiwi 설치 여부를 확인합니다.
try:
    import kiwipiepy
    print("kiwipiepy 버전:", kiwipiepy.__version__)
except ModuleNotFoundError:
    print("kiwipiepy가 현재 Notebook Python에 설치되어 있지 않습니다.")
    print("아래 설치 셀을 실행하세요.")

In [ ]:
# 현재 Notebook Python에 Kiwi가 없을 때만 이 셀의 주석을 풀어 실행합니다.
# import sys
# import subprocess
#
# subprocess.check_call([
#     sys.executable,
#     "-m",
#     "pip",
#     "install",
#     "kiwipiepy",
# ])

In [ ]:
# books_improved.csv 위치를 자동으로 찾습니다.
candidate_paths = [
    Path("notebooks/book-text-ml/books_improved.csv"),
    Path("books_improved.csv"),
]

DATA_PATH = None

for path in candidate_paths:
    if path.exists():
        DATA_PATH = path
        break

print("찾은 데이터 경로:", DATA_PATH)

if DATA_PATH is None:
    raise FileNotFoundError(
        "books_improved.csv를 찾지 못했습니다. "
        "notebooks/book-text-ml 폴더에 파일이 있는지 확인하세요."
    )

### 실습 1 결과 확인 및 정리

정상이라면 다음이 확인되어야 합니다.

- 현재 Notebook Python 경로가 표시됨
- pandas / scikit-learn import 성공
- kiwipiepy import 성공
- `books_improved.csv` 경로 확인

Kiwi가 터미널에는 설치되어 있는데 Notebook에서 import가 안 된다면 **현재 Notebook Python과 설치된 Python이 다른 것**입니다.

이 경우 `sys.executable -m pip install kiwipiepy` 방식으로 현재 커널에 설치하는 것이 가장 안전합니다.

## 실습 2. 개선 데이터 불러오기

### AI에게 질문

> books_improved.csv를 utf-8-sig로 읽고 shape, columns, 앞의 5행, 분야별 개수, 결측치, 중복 상품명을 확인하고 싶습니다.

### 초보자 상세 설명

모델을 바꾸기 전에 먼저 데이터부터 확인합니다.

수업자료 기준 개선 데이터는 **593권, 6개 분야**이며 각 분야가 약 97~100권으로 구성되어 있습니다. 실제 파일을 실행했을 때도 이 범위와 일치하는지 확인합니다.

중요한 점은 데이터가 593권으로 늘었다는 사실과 이후 성능 변화의 원인을 동일시하지 않는 것입니다. 이번 실험은 데이터 보강 효과 자체를 분리해서 측정하는 실험이 아닙니다.

In [ ]:
# 개선 데이터를 불러옵니다.
df = pd.read_csv(
    DATA_PATH,
    encoding="utf-8-sig",
)

print("데이터 크기:", df.shape)
print("컬럼:", df.columns.tolist())

display(df.head())

In [ ]:
# 필수 컬럼을 확인합니다.
required_columns = ["상품명", "분야"]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

print("누락된 필수 컬럼:", missing_columns)

if missing_columns:
    raise ValueError(
        f"필수 컬럼이 없습니다: {missing_columns}"
    )

# 상품명과 분야를 정리합니다.
df["상품명"] = (
    df["상품명"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df["분야"] = (
    df["분야"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# 빈 제목/분야는 이번 지도학습에서 사용할 수 없으므로 제외합니다.
rows_before = len(df)

df = (
    df[
        (df["상품명"] != "") &
        (df["분야"] != "")
    ]
    .reset_index(drop=True)
)

print("정리 전 행 수:", rows_before)
print("정리 후 행 수:", len(df))
print("제외된 행 수:", rows_before - len(df))

In [ ]:
# 분야별 데이터 개수를 확인합니다.
category_counts = df["분야"].value_counts()

print("분야 종류 수:", df["분야"].nunique())
display(category_counts.to_frame("도서 수"))

# 결측치와 중복 상품명을 확인합니다.
print("\n결측치:")
display(df[["상품명", "분야"]].isna().sum().to_frame("결측치"))

print("중복 상품명:", df["상품명"].duplicated().sum())

### 실습 2 결과 확인 및 정리

수업자료 기준 분야별 데이터 수는 다음과 같습니다.

- 소설: 100
- 경제/경영: 100
- 자기계발: 100
- 과학: 99
- 인문: 97
- 컴퓨터/IT: 97

총 593권입니다.

실제 출력이 다르다면 바로 성능 숫자부터 비교하지 않고 **내가 사용하는 CSV가 수업자료와 같은 버전인지 먼저 확인**합니다.

## 실습 3. Baseline용 Train/Test split 만들기

### AI에게 질문

> 상품명을 X, 분야를 y로 두고 test_size=0.2, random_state=42, stratify=y로 train/test를 나눠 주세요.  
> 이후 모든 개선 실험에서 같은 split을 재사용하고 싶습니다.

### 초보자 상세 설명

이번 Chapter에서 가장 중요한 실험 원칙 중 하나입니다.

형태소 분석 전후를 비교하면서 Train/Test가 매번 달라지면 성능 변화가 **전처리 때문인지 데이터 분할 운 때문인지** 구분하기 어렵습니다.

따라서 먼저 한 번만 split을 만들고, 이후 Baseline과 모든 Improved 실험에서 **동일한 X_train / X_test / y_train / y_test**를 사용합니다.

In [ ]:
from sklearn.model_selection import train_test_split

# 입력 X와 정답 y를 정의합니다.
X = df["상품명"]
y = df["분야"]

# 한 번만 train/test를 나눕니다.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("전체:", len(X))
print("Train:", len(X_train))
print("Test :", len(X_test))
print("Train + Test == 전체:", len(X_train) + len(X_test) == len(X))

In [ ]:
# 전체 / Train / Test의 분야 비율을 비교합니다.
split_distribution = pd.concat(
    [
        y.value_counts(normalize=True).rename("전체"),
        y_train.value_counts(normalize=True).rename("Train"),
        y_test.value_counts(normalize=True).rename("Test"),
    ],
    axis=1,
).fillna(0).mul(100).round(2)

display(split_distribution)

### 실습 3 결과 확인 및 정리

`stratify=y`를 사용하면 각 분야의 비율이 Train과 Test에 최대한 비슷하게 유지됩니다.

이제부터 개선 전후 실험에서는 **이 split을 절대 새로 만들지 않고 그대로 재사용**합니다.

## 실습 4. Baseline 모델 만들기

### AI에게 질문

> Chapter 04와 같은 구조로 Baseline을 만들고 싶습니다.
>
> 원본 제목 → 기본 TfidfVectorizer → MultinomialNB → 예측
>
> 그리고 Accuracy, Macro F1, 정답 수, classification_report를 확인해 주세요.

### 초보자 상세 설명

개선 효과를 말하려면 먼저 비교 기준이 있어야 합니다.

Baseline은 일부러 단순하게 유지합니다.

- 원본 제목 사용
- 기본 `TfidfVectorizer()`
- `MultinomialNB()`
- Train에만 `fit_transform()`
- Test에는 `transform()`만 사용

이 기준을 만든 뒤 한 번에 하나씩 전처리를 추가합니다.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
)

# Baseline TF-IDF
baseline_vectorizer = TfidfVectorizer()

X_train_baseline = baseline_vectorizer.fit_transform(
    X_train
)

X_test_baseline = baseline_vectorizer.transform(
    X_test
)

# Baseline Naive Bayes
baseline_model = MultinomialNB()

baseline_model.fit(
    X_train_baseline,
    y_train,
)

baseline_pred = baseline_model.predict(
    X_test_baseline
)

print("Train TF-IDF:", X_train_baseline.shape)
print("Test TF-IDF :", X_test_baseline.shape)

In [ ]:
# Baseline 성능을 계산합니다.
baseline_accuracy = accuracy_score(
    y_test,
    baseline_pred,
)

baseline_macro_f1 = f1_score(
    y_test,
    baseline_pred,
    average="macro",
)

baseline_correct = int(
    (baseline_pred == y_test.to_numpy()).sum()
)

print(f"Baseline Accuracy : {baseline_accuracy:.4f} ({baseline_accuracy:.2%})")
print(f"Baseline Macro F1 : {baseline_macro_f1:.4f}")
print(f"Baseline 정답 수  : {baseline_correct} / {len(y_test)}")

print("\nClassification Report")
print(
    classification_report(
        y_test,
        baseline_pred,
        zero_division=0,
    )
)

### 실습 4 자료 기준 결과

수업자료에서 동일 조건으로 측정한 Baseline은 다음과 같습니다.

- Accuracy: **48.74%**
- Macro F1: **0.4836**
- 정답: **58 / 119**

Notebook에서 실제 출력한 값이 기준입니다. 자료와 값이 다르면 코드부터 억지로 맞추지 말고 **CSV 버전, split 조건, 전처리 여부**를 먼저 확인합니다.

## 실습 5. Kiwi 형태소 분석기 만들기

### AI에게 질문

> kiwipiepy의 Kiwi를 사용해 "처음 배우는 파이썬 데이터 분석"을 tokenize하고 form과 tag를 모두 출력해 주세요.  
> NNG, NNP, SL의 의미도 정리해 주세요.

### 초보자 상세 설명

기본 TF-IDF는 문자열을 자체 규칙으로 잘라 feature를 만듭니다.

이번에는 한국어 제목에서 **의미 있는 형태소를 직접 추출한 문자열**을 TF-IDF에 넣습니다.

이번 실습에서 주로 사용할 품사는:

- `NNG` → 일반 명사
- `NNP` → 고유 명사
- `SL` → 영문

입니다.

In [ ]:
from kiwipiepy import Kiwi

kiwi = Kiwi()

sample = "처음 배우는 파이썬 데이터 분석"

tokens = kiwi.tokenize(sample)

for token in tokens:
    print(
        f"{token.form:<10}",
        token.tag,
    )

### 실습 5 결과 확인 및 정리

형태소 분석기를 썼다는 사실 자체가 개선은 아닙니다.

중요한 질문은:

> **TF-IDF에 넣는 텍스트 표현을 바꿨을 때 실제 Test 성능이 달라지는가?**

입니다.

그래서 다음 실습부터 전처리 조건을 단계별로 추가하고 매번 같은 Test에서 다시 평가합니다.

## 실습 6. 전처리 함수 3단계 만들기

이번에는 한 번에 많은 규칙을 넣지 않고 세 단계를 분리합니다.

### 단계 A — 형태소 분석

- NNG / NNP / SL만 사용
- 영문 대소문자 차이를 줄이기 위해 소문자 변환

### 단계 B — 짧은 토큰·숫자 필터

- 2글자 미만 제외
- 숫자로만 구성된 토큰 제외

### 단계 C — 최소 불용어

- `에디션` 제외

분야 핵심 단어인 파이썬, 데이터, 분석, AI, 인공지능, 머신러닝, SQL, 경제, 투자, 과학, 철학, 소설 등은 제거하지 않습니다.

In [ ]:
# 형태소와 필터 기준입니다.
TARGET_TAGS = {"NNG", "NNP", "SL"}
MIN_LENGTH = 2
STOPWORDS = {"에디션"}


def tokenize_pos_only(text):
    """A단계: 품사만 골라 형태소 문자열을 만듭니다."""
    words = []

    for token in kiwi.tokenize(str(text)):
        if token.tag not in TARGET_TAGS:
            continue

        word = token.form.strip().lower()

        if word:
            words.append(word)

    return " ".join(words)


def tokenize_with_short_filter(text):
    """B단계: A단계 + 짧은 토큰과 숫자 토큰을 제외합니다."""
    words = []

    for token in kiwi.tokenize(str(text)):
        if token.tag not in TARGET_TAGS:
            continue

        word = token.form.strip().lower()

        if len(word) < MIN_LENGTH:
            continue

        if word.isdigit():
            continue

        words.append(word)

    return " ".join(words)


def preprocess_title(text):
    """C단계: B단계 + 최소 불용어 필터를 적용합니다."""
    words = []

    for token in kiwi.tokenize(str(text)):
        if token.tag not in TARGET_TAGS:
            continue

        word = token.form.strip().lower()

        if len(word) < MIN_LENGTH:
            continue

        if word.isdigit():
            continue

        if word in STOPWORDS:
            continue

        words.append(word)

    return " ".join(words)

In [ ]:
# 한 문장에 세 전처리를 적용해 차이를 확인합니다.
sample_title = "2026 에디션 AI 시대의 파이썬 데이터 분석"

print("원본:")
print(sample_title)

print("\nA. 형태소 분석:")
print(tokenize_pos_only(sample_title))

print("\nB. 짧은/숫자 필터:")
print(tokenize_with_short_filter(sample_title))

print("\nC. + 불용어:")
print(preprocess_title(sample_title))

### 실습 6 결과 확인 및 정리

전처리를 한 번에 모두 적용하면 어떤 변경이 실제 성능에 영향을 주었는지 알기 어렵습니다.

이번에는:

**Baseline → A → B → C**

순서로 하나씩 추가해서 성능을 비교합니다.

이 방식이 “불용어를 많이 넣어 정답을 맞추는 것”보다 실험 결과를 설명하기 훨씬 쉽습니다.

## 실습 7. 전체 제목의 전처리 전후 확인하기

### AI에게 질문

> 전체 상품명에 형태소 분석/짧은 토큰 필터/최종 전처리를 적용하고 원본과 나란히 앞의 10개를 확인해 주세요.

### 초보자 상세 설명

전처리 함수는 성능 숫자만 보고 믿지 않습니다.

실제 제목이 어떤 문자열로 변했는지 **사람이 직접 몇 개 읽어보는 검증**이 필요합니다.

특히 핵심 분야 단어가 사라지지 않았는지 확인합니다.

In [ ]:
# 전체 데이터에 단계별 전처리를 적용합니다.
df["상품명_형태소"] = df["상품명"].apply(
    tokenize_pos_only
)

df["상품명_짧은필터"] = df["상품명"].apply(
    tokenize_with_short_filter
)

df["상품명_정제"] = df["상품명"].apply(
    preprocess_title
)

display(
    df[
        [
            "상품명",
            "상품명_형태소",
            "상품명_짧은필터",
            "상품명_정제",
            "분야",
        ]
    ].head(10)
)

In [ ]:
# 전처리 후 완전히 빈 문자열이 된 제목 수를 확인합니다.
for col in [
    "상품명_형태소",
    "상품명_짧은필터",
    "상품명_정제",
]:
    empty_count = int(
        (df[col].str.len() == 0).sum()
    )

    print(
        f"{col} 빈 문자열 수:",
        empty_count,
    )

### 실습 7 결과 확인 및 정리

전처리 후 빈 제목이 생기면 해당 제목은 TF-IDF에서 정보가 거의 없는 0 벡터가 될 수 있습니다.

따라서 “전처리를 더 많이 했다 = 무조건 더 좋다”가 아닙니다.

정보를 너무 많이 제거하면 오히려 성능이 떨어질 수 있습니다.

## 실습 8. 동일 split에 단계별 전처리 적용하기

중요한 부분입니다.

전체 데이터에 전처리 컬럼을 만들어 놓았지만, 성능 비교에서는 **기존 X_train / X_test의 index를 그대로 사용**해야 합니다.

그래야 Baseline과 개선 모델이 정확히 같은 책을 Train/Test로 사용합니다.

In [ ]:
# 기존 split의 index를 그대로 이용합니다.
train_idx = X_train.index
test_idx = X_test.index

train_pos = df.loc[
    train_idx,
    "상품명_형태소",
]

test_pos = df.loc[
    test_idx,
    "상품명_형태소",
]

train_short = df.loc[
    train_idx,
    "상품명_짧은필터",
]

test_short = df.loc[
    test_idx,
    "상품명_짧은필터",
]

train_final = df.loc[
    train_idx,
    "상품명_정제",
]

test_final = df.loc[
    test_idx,
    "상품명_정제",
]

print("Baseline Train:", len(X_train))
print("A Train       :", len(train_pos))
print("B Train       :", len(train_short))
print("C Train       :", len(train_final))

print("\n모든 Train 길이 동일:",
      len(X_train) == len(train_pos) == len(train_short) == len(train_final))

print("모든 Test 길이 동일:",
      len(X_test) == len(test_pos) == len(test_short) == len(test_final))

## 실습 9. 단계별 모델 평가 함수 만들기

### AI에게 질문

> 같은 MultinomialNB를 유지한 채 텍스트 표현만 바꾸어 Accuracy, Macro F1, 정답 수를 비교하고 싶습니다.  
> train_texts, test_texts, y_train, y_test를 받아 결과와 모델을 반환하는 함수를 만들어 주세요.

### 초보자 상세 설명

알고리즘까지 동시에 바꾸면 어떤 변화가 성능에 영향을 주었는지 구분하기 어렵습니다.

이번 Chapter에서는 **모델은 그대로 두고 텍스트 표현만 바꿉니다.**

In [ ]:
def evaluate_text_representation(
    train_texts,
    test_texts,
    y_train,
    y_test,
):
    # 각 실험마다 새로운 Vectorizer를 사용합니다.
    vectorizer = TfidfVectorizer()

    X_train_vec = vectorizer.fit_transform(
        train_texts
    )

    X_test_vec = vectorizer.transform(
        test_texts
    )

    # 알고리즘은 모든 실험에서 동일하게 유지합니다.
    model = MultinomialNB()

    model.fit(
        X_train_vec,
        y_train,
    )

    pred = model.predict(
        X_test_vec
    )

    accuracy = accuracy_score(
        y_test,
        pred,
    )

    macro_f1 = f1_score(
        y_test,
        pred,
        average="macro",
    )

    correct = int(
        (pred == y_test.to_numpy()).sum()
    )

    return {
        "vectorizer": vectorizer,
        "model": model,
        "pred": pred,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "correct": correct,
        "train_shape": X_train_vec.shape,
        "test_shape": X_test_vec.shape,
    }

## 실습 10. 형태소 분석만 적용한 결과 측정하기

A단계에서는 NNG / NNP / SL 형태소 선택만 적용합니다.

In [ ]:
result_pos = evaluate_text_representation(
    train_pos,
    test_pos,
    y_train,
    y_test,
)

print(f"형태소 Accuracy: {result_pos['accuracy']:.4f} ({result_pos['accuracy']:.2%})")
print(f"형태소 Macro F1: {result_pos['macro_f1']:.4f}")
print(f"형태소 정답 수 : {result_pos['correct']} / {len(y_test)}")

### 실습 10 자료 기준 관찰

수업자료에서는 형태소 분석 단계에서 Baseline보다 **정답이 1권 증가**했습니다.

중요한 것은 “Kiwi를 썼으니 좋아졌다”가 아니라 **동일 Test에서 실제 숫자가 어떻게 변했는지**입니다.

## 실습 11. 짧은 토큰 필터까지 적용하기

B단계에서는 형태소 분석 결과에서:

- 2글자 미만 제외
- 숫자로만 된 토큰 제외

를 추가합니다.

In [ ]:
result_short = evaluate_text_representation(
    train_short,
    test_short,
    y_train,
    y_test,
)

print(f"짧은필터 Accuracy: {result_short['accuracy']:.4f} ({result_short['accuracy']:.2%})")
print(f"짧은필터 Macro F1: {result_short['macro_f1']:.4f}")
print(f"짧은필터 정답 수 : {result_short['correct']} / {len(y_test)}")

### 실습 11 자료 기준 관찰

수업자료에서는 짧은 토큰 필터까지 적용한 결과가 Baseline보다 **정답 2권 증가** 단계로 기록되어 있습니다.

Notebook에서는 현재 CSV로 직접 계산한 숫자를 기준으로 판단합니다.

## 실습 12. 불용어 '에디션'까지 적용하기

C단계에서는 최소 불용어 `에디션`을 추가로 제외합니다.

불용어 목록을 크게 만들지 않는 이유는 특정 데이터의 정답에 맞추기 위해 단어를 많이 제거하면 **일반화가 아니라 결과 맞추기**가 될 수 있기 때문입니다.

In [ ]:
result_final = evaluate_text_representation(
    train_final,
    test_final,
    y_train,
    y_test,
)

print(f"Improved Accuracy: {result_final['accuracy']:.4f} ({result_final['accuracy']:.2%})")
print(f"Improved Macro F1: {result_final['macro_f1']:.4f}")
print(f"Improved 정답 수 : {result_final['correct']} / {len(y_test)}")

### 실습 12 자료 기준 관찰

수업자료에서는 `에디션` 제거 자체가 **추가 성능 향상을 만들지는 않았습니다.**

이것도 중요한 결과입니다.

> 전처리 규칙을 추가했다고 성능이 반드시 좋아지는 것은 아닙니다.

## 실습 13. Baseline → Improved 한 표로 비교하기

이제 네 실험을 한 표에 모읍니다.

- Baseline
- 형태소 분석
- 짧은 토큰 필터
- 최종(에디션 제외)

In [ ]:
comparison = pd.DataFrame([
    {
        "단계": "Baseline",
        "Accuracy": baseline_accuracy,
        "Macro_F1": baseline_macro_f1,
        "정답수": baseline_correct,
    },
    {
        "단계": "형태소 분석",
        "Accuracy": result_pos["accuracy"],
        "Macro_F1": result_pos["macro_f1"],
        "정답수": result_pos["correct"],
    },
    {
        "단계": "짧은 토큰 필터",
        "Accuracy": result_short["accuracy"],
        "Macro_F1": result_short["macro_f1"],
        "정답수": result_short["correct"],
    },
    {
        "단계": "최종 Improved",
        "Accuracy": result_final["accuracy"],
        "Macro_F1": result_final["macro_f1"],
        "정답수": result_final["correct"],
    },
])

comparison["Accuracy(%)"] = (
    comparison["Accuracy"] * 100
).round(2)

comparison["Macro_F1"] = (
    comparison["Macro_F1"]
    .round(4)
)

display(
    comparison[
        [
            "단계",
            "Accuracy(%)",
            "Macro_F1",
            "정답수",
        ]
    ]
)

In [ ]:
# Baseline 대비 최종 변화량을 계산합니다.
accuracy_change_pp = (
    result_final["accuracy"]
    - baseline_accuracy
) * 100

macro_f1_change = (
    result_final["macro_f1"]
    - baseline_macro_f1
)

correct_change = (
    result_final["correct"]
    - baseline_correct
)

print(f"Accuracy 변화: {accuracy_change_pp:+.2f}%p")
print(f"Macro F1 변화: {macro_f1_change:+.4f}")
print(f"정답 수 변화 : {correct_change:+d}권")

### 실습 13 자료 기준 Before / After

수업자료 기준:

| Model | Accuracy | Macro F1 | 정답 |
|---|---:|---:|---:|
| Baseline | 48.74% | 0.4836 | 58 / 119 |
| Improved | 51.26% | 0.5047 | 61 / 119 |

Accuracy는 약 **2.52%p 상승**, 정답은 **3권 증가**입니다.

따라서 자료의 결론은 “대폭 향상”이 아니라:

> **개선 효과는 확인되었지만 크지는 않다.**

입니다.

## 실습 14. Improved Classification Report 확인하기

Accuracy 하나만 보고 끝내지 않고 분야별 precision / recall / F1-score도 확인합니다.

In [ ]:
print(
    classification_report(
        y_test,
        result_final["pred"],
        zero_division=0,
    )
)

In [ ]:
# 분야별 report를 DataFrame으로도 확인합니다.
improved_report = classification_report(
    y_test,
    result_final["pred"],
    zero_division=0,
    output_dict=True,
)

improved_report_df = (
    pd.DataFrame(improved_report)
    .T
)

display(improved_report_df.round(4))

### 실습 14 결과 확인 및 정리

전체 Accuracy가 올라도 모든 분야가 동일하게 좋아지는 것은 아닙니다.

따라서 어떤 분야는 개선되고 어떤 분야는 그대로이거나 나빠졌는지도 확인합니다.

이번 Chapter의 핵심은 **숫자 하나를 높이는 것보다 변경의 효과를 실제 평가값으로 확인하는 것**입니다.

## 실습 15. Baseline과 Improved가 서로 다르게 예측한 책 확인하기

전체 Test에서 두 모델의 예측이 달라진 제목을 직접 읽어봅니다.

In [ ]:
prediction_compare = pd.DataFrame({
    "상품명": X_test.to_numpy(),
    "실제_분야": y_test.to_numpy(),
    "Baseline_예측": baseline_pred,
    "Improved_예측": result_final["pred"],
})

prediction_compare["Baseline_정답"] = (
    prediction_compare["실제_분야"]
    == prediction_compare["Baseline_예측"]
)

prediction_compare["Improved_정답"] = (
    prediction_compare["실제_분야"]
    == prediction_compare["Improved_예측"]
)

changed_predictions = prediction_compare[
    prediction_compare["Baseline_예측"]
    != prediction_compare["Improved_예측"]
].copy()

print("예측이 달라진 책 수:", len(changed_predictions))

display(changed_predictions.head(30))

In [ ]:
# Baseline은 틀리고 Improved는 맞은 경우를 확인합니다.
fixed_cases = prediction_compare[
    (~prediction_compare["Baseline_정답"]) &
    (prediction_compare["Improved_정답"])
].copy()

# Baseline은 맞고 Improved는 틀린 경우도 확인합니다.
broken_cases = prediction_compare[
    (prediction_compare["Baseline_정답"]) &
    (~prediction_compare["Improved_정답"])
].copy()

print("오답 → 정답:", len(fixed_cases))
print("정답 → 오답:", len(broken_cases))

print("\n오답 → 정답 사례")
display(fixed_cases.head(20))

print("\n정답 → 오답 사례")
display(broken_cases.head(20))

### 실습 15 결과 확인 및 정리

개선 모델은 기존 오답을 일부 고칠 수 있지만, 반대로 기존 정답을 틀리게 만드는 경우도 있을 수 있습니다.

그래서 특정 성공 사례 하나만 골라 “개선됐다”고 말하지 않습니다.

**전체 Test의 Accuracy / Macro F1 / 정답 수와 함께 사례를 봅니다.**

## 실습 16. 대표 제목 '처음 배우는 파이썬 데이터 분석' 확인하기

수업자료에서는 이 제목이 593권 데이터에서 Baseline과 Improved 모두 **컴퓨터/IT**로 예측되었습니다.

따라서 이 제목을 “전처리 덕분에 오답이 정답으로 바뀐 사례”라고 설명하면 안 됩니다.

In [ ]:
representative_title = "처음 배우는 파이썬 데이터 분석"

# Baseline 예측
baseline_vector = baseline_vectorizer.transform(
    [representative_title]
)

baseline_example_pred = baseline_model.predict(
    baseline_vector
)[0]

# Improved 예측
clean_representative_title = preprocess_title(
    representative_title
)

improved_vector = result_final["vectorizer"].transform(
    [clean_representative_title]
)

improved_example_pred = result_final["model"].predict(
    improved_vector
)[0]

print("원본 제목:", representative_title)
print("정제 제목:", clean_representative_title)
print("Baseline 예측:", baseline_example_pred)
print("Improved 예측:", improved_example_pred)

### 실습 16 결과 확인 및 정리

같은 알고리즘이라도 학습 데이터가 달라지면 결과가 달라질 수 있습니다.

따라서 특정 제목 한 권만 보고 전체 모델의 품질을 판단하면 안 됩니다.

**개별 사례는 설명용이고, 전체 Test 평가는 판단용**이라고 구분하면 좋습니다.

## 실습 17. Baseline 추천 문제 재현하기

기존 추천은:

**전체 도서 → 원본 제목 TF-IDF → Cosine Similarity → 무조건 Top 5**

였습니다.

이 방식은 공통 feature가 없어 유사도가 0이어도 정렬 순위에 따라 5권을 채울 수 있습니다.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Baseline 추천용 행렬: 원본 제목 전체
baseline_reco_vectorizer = TfidfVectorizer()

baseline_reco_matrix = baseline_reco_vectorizer.fit_transform(
    df["상품명"]
)


def baseline_recommend(
    selected_index,
    top_n=5,
):
    scores = cosine_similarity(
        baseline_reco_matrix[
            selected_index:selected_index + 1
        ],
        baseline_reco_matrix,
    ).ravel()

    scores[selected_index] = -1

    top_indices = (
        scores
        .argsort()[::-1][:top_n]
    )

    result = df.iloc[
        top_indices
    ][["상품명", "분야"]].copy()

    result["similarity"] = (
        scores[top_indices]
        .round(4)
    )

    return result.reset_index(drop=True)

In [ ]:
# 수업자료 대표 사례인 "거짓말이 달려온다"를 우선 찾습니다.
target_title = "거짓말이 달려온다"

matches = df.index[
    df["상품명"] == target_title
].tolist()

if matches:
    recommendation_test_index = matches[0]
else:
    recommendation_test_index = 0
    print(
        f"'{target_title}'가 없어 index 0 도서로 대신 확인합니다."
    )

print("기준 도서:")
display(
    df.loc[
        [recommendation_test_index],
        ["상품명", "분야"],
    ]
)

print("\nBaseline 추천:")
baseline_reco_example = baseline_recommend(
    recommendation_test_index,
    top_n=5,
)

display(baseline_reco_example)

In [ ]:
# Baseline 추천에 유사도 0인 도서가 포함되었는지 확인합니다.
zero_similarity_count = int(
    (baseline_reco_example["similarity"] <= 0).sum()
)

print(
    "Baseline Top5 중 유사도 0 이하:",
    zero_similarity_count,
)

### 실습 17 결과 확인 및 정리

유사도 0인 책은 현재 TF-IDF 표현에서 기준 도서와 **공통된 방향 정보가 없다는 뜻**입니다.

그런데도 Top 5 개수를 채우기 위해 출력하면 사용자 입장에서는 “왜 이 책이 추천됐지?”라는 문제가 생깁니다.

이것은 Naive Bayes나 TF-IDF 모델 자체보다 **추천 결과를 후처리하는 로직의 문제**입니다.

## 실습 18. Improved 추천용 텍스트 만들기

개선 추천은 다음 흐름으로 바꿉니다.

**선택 도서 → 같은 분야 후보 → 형태소 분석 + 단어 필터 → TF-IDF → Cosine Similarity → 자기 자신 제외 → similarity > 0 → 최대 Top 5**

후보가 5권보다 적으면 0점짜리 책으로 억지로 채우지 않습니다.

In [ ]:
# 최종 전처리된 제목 전체로 추천용 TF-IDF를 만듭니다.
improved_reco_vectorizer = TfidfVectorizer()

improved_reco_matrix = (
    improved_reco_vectorizer
    .fit_transform(
        df["상품명_정제"]
    )
)

print(
    "Improved 추천 matrix:",
    improved_reco_matrix.shape,
)

## 실습 19. Improved 추천 함수 만들기

### AI에게 질문

> 선택 도서와 같은 분야만 후보로 사용하고, 자기 자신과 동일 제목을 제외하고, cosine similarity가 0보다 큰 책만 최대 5권 반환하는 함수를 만들어 주세요. 후보가 없으면 빈 DataFrame을 반환하게 해 주세요.

In [ ]:
def improved_recommend(
    selected_index,
    top_n=5,
):
    if not 0 <= selected_index < len(df):
        raise IndexError(
            f"유효하지 않은 index입니다: {selected_index}"
        )

    selected_title = df.loc[
        selected_index,
        "상품명",
    ]

    selected_category = df.loc[
        selected_index,
        "분야",
    ]

    # 선택 도서와 같은 분야의 index만 후보로 사용합니다.
    candidate_indices = df.index[
        df["분야"] == selected_category
    ].to_numpy()

    # 선택 도서와 전체 도서의 유사도를 계산합니다.
    scores = cosine_similarity(
        improved_reco_matrix[
            selected_index:selected_index + 1
        ],
        improved_reco_matrix,
    ).ravel()

    recommended_indices = []

    # 같은 분야 후보만 유사도 높은 순서로 정렬합니다.
    sorted_candidates = candidate_indices[
        np.argsort(
            scores[candidate_indices]
        )[::-1]
    ]

    for idx in sorted_candidates:
        idx = int(idx)

        # 자기 자신 제외
        if idx == selected_index:
            continue

        # 동일 제목 중복 제외
        if df.loc[idx, "상품명"] == selected_title:
            continue

        # 유사도 0 이하 제외
        if scores[idx] <= 0:
            continue

        recommended_indices.append(idx)

        if len(recommended_indices) >= top_n:
            break

    display_columns = [
        col
        for col in [
            "상품명",
            "저자",
            "출판사",
            "분야",
        ]
        if col in df.columns
    ]

    result = (
        df.loc[
            recommended_indices,
            display_columns,
        ]
        .copy()
        .reset_index(drop=True)
    )

    result["similarity"] = [
        round(
            float(scores[idx]),
            4,
        )
        for idx in recommended_indices
    ]

    return result

## 실습 20. 추천 Before / After 비교하기

같은 기준 도서로 Baseline과 Improved 결과를 나란히 확인합니다.

In [ ]:
print("기준 도서")
display(
    df.loc[
        [recommendation_test_index],
        ["상품명", "분야"],
    ]
)

print("\nBaseline")
display(
    baseline_recommend(
        recommendation_test_index,
        top_n=5,
    )
)

print("\nImproved")
improved_reco_example = improved_recommend(
    recommendation_test_index,
    top_n=5,
)

if improved_reco_example.empty:
    print(
        "현재 기준으로 유사도가 있는 추천 도서를 찾지 못했습니다."
    )
else:
    display(improved_reco_example)

### 실습 20 자료 기준 대표 사례

수업자료에서는 **“거짓말이 달려온다”**를 대표 사례로 확인했습니다.

- Baseline → 유사도 0인 도서 5권 출력
- Improved → 양수 유사도 후보 없음 → 추천 없음 안내

이 결과에서 중요한 것은 “추천을 못 했다”가 실패가 아니라는 점입니다.

> **근거 없는 0점 추천을 하지 않는 것이 더 올바른 결과일 수 있습니다.**

## 실습 21. Improved 추천 조건 자동 검증하기

함수 결과가 우리가 정한 규칙을 실제로 지키는지 검사합니다.

In [ ]:
def validate_recommendation(
    selected_index,
    recommendations,
):
    selected_title = df.loc[
        selected_index,
        "상품명",
    ]

    selected_category = df.loc[
        selected_index,
        "분야",
    ]

    checks = {
        "최대 5권": len(recommendations) <= 5,
        "자기/동일 제목 제외": (
            recommendations.empty
            or not (
                recommendations["상품명"]
                == selected_title
            ).any()
        ),
        "같은 분야만 추천": (
            recommendations.empty
            or (
                recommendations["분야"]
                == selected_category
            ).all()
        ),
        "유사도 > 0": (
            recommendations.empty
            or (
                recommendations["similarity"]
                > 0
            ).all()
        ),
        "유사도 내림차순": (
            len(recommendations) < 2
            or np.all(
                recommendations["similarity"].to_numpy()[:-1]
                >= recommendations["similarity"].to_numpy()[1:]
            )
        ),
    }

    return pd.DataFrame(
        {
            "검사항목": list(checks.keys()),
            "통과여부": list(checks.values()),
        }
    )


display(
    validate_recommendation(
        recommendation_test_index,
        improved_reco_example,
    )
)

## 실습 22. 전체 도서에서 추천 없음 비율 측정하기

추천 로직을 엄격하게 만들면 의미 없는 추천은 줄지만 추천 가능한 책 수도 줄어듭니다.

수업자료에서는 593권 중 **229권**이 개선 조건에서 추천 결과가 없었고, 약 **38.6%**였습니다.

현재 CSV에서도 직접 계산합니다.

In [ ]:
# 전체 도서를 한 번씩 기준으로 잡아 추천 가능 여부를 확인합니다.
no_recommendation_indices = []

recommendation_counts = []

for idx in range(len(df)):
    result = improved_recommend(
        idx,
        top_n=5,
    )

    recommendation_counts.append(
        len(result)
    )

    if result.empty:
        no_recommendation_indices.append(
            idx
        )

no_reco_count = len(
    no_recommendation_indices
)

no_reco_ratio = (
    no_reco_count
    / len(df)
)

print(
    "전체 도서 수:",
    len(df),
)

print(
    "추천 결과가 없는 도서 수:",
    no_reco_count,
)

print(
    f"추천 없음 비율: {no_reco_ratio:.2%}",
)

In [ ]:
# 추천 가능한 개수 분포도 확인합니다.
recommendation_count_series = pd.Series(
    recommendation_counts,
    name="추천가능수",
)

display(
    recommendation_count_series
    .value_counts()
    .sort_index()
    .to_frame("도서 수")
)

### 실습 22 결과 해석

기존 방식은 거의 항상 5권을 보여줄 수 있지만 유사도 0도 포함할 수 있습니다.

개선 방식은:

- **추천 개수는 줄 수 있음**
- 대신 **유사도 0인 근거 없는 추천을 제거**

합니다.

즉 추천에서는 **Coverage(얼마나 많이 추천 가능한가)**와 **추천 근거의 품질** 사이에 trade-off가 생깁니다.

## 실습 23. Streamlit 앱에 들어갈 최종 분류 파이프라인 확인하기

앱에서는 Test 평가를 다시 하지 않습니다.

Chapter 07에서 평가가 끝났으므로 앱에서는 사용 가능한 전체 라벨 데이터로 **최종 시연용 Improved 모델**을 학습합니다.

새 제목은:

**사용자 입력 → preprocess_title() → 기존 TF-IDF transform() → Naive Bayes predict()**

순서입니다.

In [ ]:
# 앱과 같은 방식으로 최종 분류 모델을 준비해 봅니다.
app_vectorizer = TfidfVectorizer()

app_X = app_vectorizer.fit_transform(
    df["상품명_정제"]
)

app_y = df["분야"]

app_model = MultinomialNB()

app_model.fit(
    app_X,
    app_y,
)

# 새 제목 예측 확인
app_test_title = "처음 배우는 파이썬 데이터 분석"
app_clean_title = preprocess_title(
    app_test_title
)

app_test_vector = app_vectorizer.transform(
    [app_clean_title]
)

app_prediction = app_model.predict(
    app_test_vector
)[0]

print("입력:", app_test_title)
print("정제:", app_clean_title)
print("예상 분야:", app_prediction)

## 실습 24. Chapter 07 앱 최종 검증 체크

GitHub의 `app.py`도 이번 Chapter 기준으로 수정합니다.

앱에서 직접 확인할 항목:

- `books_improved.csv`가 우선 사용되는가?
- 제목 입력 시 Kiwi 전처리가 적용되는가?
- 새 제목에는 `transform()`만 사용하는가?
- 예상 분야가 출력되는가?
- 기준 도서를 선택할 수 있는가?
- 추천 도서가 같은 분야인가?
- 자기 자신/동일 제목이 제외되는가?
- 유사도 0인 도서가 제외되는가?
- 후보가 없으면 “추천 없음” 메시지가 보이는가?
- 터미널에 실행 오류가 없는가?

## 실습 25. 현재 모델의 한계

현재 모델은 거의 **도서 제목만 사용**합니다.

사용하지 않는 정보:

- 책 소개
- 목차
- 본문
- 키워드
- 독자 행동
- 구매 이력
- 평점
- 개인 취향

따라서 제목만으로 6개 분야를 완벽하게 분류하거나 내용·취향까지 반영한 추천을 만드는 데는 한계가 있습니다.

이번 Chapter의 목적은 이 한계를 숨기는 것이 아니라 **숫자와 실제 결과로 확인하는 것**입니다.

## 실습 26. Chapter 07 최종 결과 Markdown 만들기

아래 셀은 현재 실제 실행값을 사용해 최종 비교 Markdown을 만듭니다.

자료에 적힌 숫자를 무조건 복사하지 않고 **내 실행 결과를 자동으로 넣습니다.**

In [ ]:
from IPython.display import Markdown, display

final_markdown = f"""
## Chapter 07 결과

### 데이터

- 전체 도서: **{len(df)}권**
- 분야 수: **{df["분야"].nunique()}개**
- Train: **{len(X_train)}권**
- Test: **{len(X_test)}권**

### 분류 Before / After

- Baseline Accuracy: **{baseline_accuracy:.2%}**
- Improved Accuracy: **{result_final["accuracy"]:.2%}**
- Accuracy 변화: **{accuracy_change_pp:+.2f}%p**

- Baseline Macro F1: **{baseline_macro_f1:.4f}**
- Improved Macro F1: **{result_final["macro_f1"]:.4f}**

- Baseline 정답: **{baseline_correct} / {len(y_test)}**
- Improved 정답: **{result_final["correct"]} / {len(y_test)}**
- 정답 변화: **{correct_change:+d}권**

### 추천 개선

- 전체 도서: **{len(df)}권**
- Improved 조건에서 추천 결과가 없는 도서: **{no_reco_count}권**
- 추천 없음 비율: **{no_reco_ratio:.2%}**

### 나의 판단

형태소 분석과 최소 필터링이 실제 평가 결과에 어떤 영향을 주는지
동일한 Train/Test 조건에서 비교했습니다.

추천에서는 항상 5권을 채우는 것보다
유사도 0인 도서를 제외해 추천 근거를 유지하는 방향으로 개선했습니다.

### 한계

현재 분류와 추천은 주로 도서 제목 텍스트에 의존합니다.
책 소개, 본문, 사용자 행동과 같은 정보는 사용하지 않았습니다.
"""

display(
    Markdown(final_markdown)
)

## 확인 문제

1. Baseline을 먼저 만드는 이유는 무엇인가?
2. 형태소 분석 전후 성능을 같은 Train/Test 데이터에서 비교해야 하는 이유는 무엇인가?
3. 형태소 분석과 짧은 토큰 필터가 각각 실제 결과에 어떤 영향을 주었는가?
4. `에디션` 제거가 추가 성능 향상을 만들었는가?
5. 코사인 유사도 0인 책을 추천에서 제외한 이유는 무엇인가?
6. 추천 결과가 5권 미만일 때 0점 책으로 채우지 않는 이유는 무엇인가?
7. 특정 제목 하나만으로 전체 모델 개선을 판단하면 안 되는 이유는 무엇인가?
8. 현재 추천이 책 내용이나 개인 취향의 유사성을 보장하지 못하는 이유는 무엇인가?

## 최종 체크리스트

- [ ] books_improved.csv를 불러왔다.
- [ ] 실제 데이터 크기와 6개 분야 분포를 확인했다.
- [ ] 결측치와 중복 상품명을 확인했다.
- [ ] Train/Test를 한 번만 분리했다.
- [ ] Baseline Accuracy / Macro F1 / 정답 수를 기록했다.
- [ ] Kiwi token 결과를 직접 확인했다.
- [ ] NNG / NNP / SL 기준을 확인했다.
- [ ] 형태소 분석만 적용한 성능을 측정했다.
- [ ] 짧은 토큰·숫자 필터 적용 후 다시 측정했다.
- [ ] 최소 불용어 적용 후 다시 측정했다.
- [ ] Baseline과 Improved를 같은 표에서 비교했다.
- [ ] Classification Report를 확인했다.
- [ ] 오답→정답과 정답→오답 사례를 모두 확인했다.
- [ ] Baseline 추천의 유사도 0 문제를 확인했다.
- [ ] 같은 분야 후보 + similarity > 0 추천을 구현했다.
- [ ] 후보가 없을 때 빈 추천 결과를 정상으로 처리했다.
- [ ] 전체 데이터에서 추천 없음 비율을 측정했다.
- [ ] Streamlit app.py에 동일한 전처리·추천 규칙을 연결했다.
- [ ] 특정 예제 하나가 아니라 전체 평가 결과로 개선을 판단했다.
- [ ] 현재 모델의 한계를 설명할 수 있다.

## Chapter 07 한 문장 정리

**실행되는 모델을 그대로 믿지 않고, Baseline을 먼저 측정한 뒤 한국어 형태소 분석과 작은 필터링 변경을 같은 조건에서 검증하고, 유사도 0인 근거 없는 추천을 제거하여 분류와 추천 결과를 한 단계 더 신뢰할 수 있게 개선합니다.**